# Exercise 04: Logistic Regression and Model Evaluation – Customer Churn

**Dataset:** 7'043 customers of a fictional telecom provider (IBM sample data set based on realistic customer
structures). For each customer we know the contract, the booked services, the charges and whether the customer
**left the company in the last month** (`Churn`). Source:
[Kaggle – Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).

| variable | meaning |
|---|---|
| `tenure` | number of months the customer has stayed with the company |
| `Contract` | Month-to-month, One year, Two year |
| `InternetService` | DSL, Fiber optic, No |
| `MonthlyCharges`, `TotalCharges` | charges per month / in total (USD) |
| `PaymentMethod`, `PaperlessBilling`, `TechSupport`, `OnlineSecurity`, … | further contract and service details |
| `Churn` | Yes / No – **target** |

**Business question:** Which customers are likely to cancel their contract? The marketing team wants to send a
retention offer to customers at risk. Since an offer costs money and a lost customer costs even more, the
**classification threshold** must be chosen carefully.

**Slides:** Part 04 «Classification models» – logistic regression, evaluation (confusion matrix, metrics, thresholds,
ROC curve and AUC).

**After this exercise you can ...**
- compute probabilities of a logistic regression model by hand,
- fit and interpret a logistic regression in scikit-learn (including categorical features),
- compute and interpret accuracy, precision, recall, F1 and FPR,
- choose a threshold based on costs of false positives and false negatives,
- draw and interpret a ROC curve and the AUC.

## Concept

**Logistic regression.** A linear score is squeezed by the sigmoid function into a probability:

$$P(y=1 \mid x) = \frac{1}{1+e^{-(\beta_0+\beta_1 x_1+\dots+\beta_p x_p)}}$$

$\beta_j > 0$: the probability rises with $x_j$. The model is fitted by maximum likelihood (minimising the log-loss).
(Optional: $e^{\beta_j}$ is the factor by which the *odds* $p/(1-p)$ change per unit of $x_j$.)

**Threshold.** Predict "positive" if $\hat p \ge \tau$ (default $\tau = 0.5$). The threshold should be chosen based on
the costs of false positives vs. false negatives.

**Confusion matrix and metrics.**

| | actual positive | actual negative |
|---|---|---|
| **predicted positive** | TP | FP |
| **predicted negative** | FN | TN |

$$\text{Accuracy}=\frac{TP+TN}{n},\quad \text{Precision}=\frac{TP}{TP+FP},\quad
\text{Recall (TPR)}=\frac{TP}{TP+FN},\quad \text{FPR}=\frac{FP}{FP+TN},\quad
F_1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

With imbalanced classes, accuracy alone misleads.

**ROC curve and AUC.** For every threshold $\tau$ plot (FPR, TPR). AUC = area under the curve = probability that a
random positive case gets a higher score than a random negative case (0.5 = random ranking, 1 = perfect).

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, roc_auc_score, classification_report)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Probabilities.** A logistic regression with the only feature `tenure` (months with the company) has the
coefficients $\beta_0 = -0.5$ and $\beta_1 = -0.08$. Compute the churn probability for customers with a tenure of
1, 12 and 48 months. Which of them are classified as churners with $\tau = 0.5$ and with $\tau = 0.2$?

**b) Metrics.** A churn model is applied to 1'000 customers, 180 of whom actually churn. It flags 200 customers as
churners, of whom 120 really churn. Fill in the confusion matrix and compute accuracy, precision, recall, FPR and F1.

**c) Baseline.** What accuracy does a "model" reach that simply predicts "no churn" for every customer? What is its
recall?

**d)** Verify your results with Python.

In [ ]:
# a) Probabilities
beta_0, beta_1 = -0.5, -0.08
tenure = np.array([1, 12, 48])
score = beta_0 + beta_1 * tenure
p = 1 / (1 + np.exp(-score))
for t, s, pi in zip(tenure, score, p):
    print(f'tenure = {t:2d}: score = {s:5.2f}, p = {pi:.3f}, churn (tau=0.5): {pi >= 0.5}, churn (tau=0.2): {pi >= 0.2}')

# b) Metrics
TP, FP, FN, TN = 120, 80, 60, 740
n = TP + FP + FN + TN
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print(f'\nAccuracy  = {(TP + TN) / n:.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall    = {recall:.3f}')
print(f'FPR       = {FP / (FP + TN):.3f}')
print(f'F1        = {2 * precision * recall / (precision + recall):.3f}')

# c) Baseline: always predict 'no churn'
print(f'\nBaseline accuracy = {(FP + TN) / n:.3f}, baseline recall = 0')

**Solution:**

a) Scores: −0.58, −1.46, −4.34 → $p = 1/(1+e^{0.58}) \approx 0.359$, $p \approx 0.188$, $p \approx 0.013$.
With $\tau = 0.5$ nobody is classified as a churner; with $\tau = 0.2$ only the new customer (1 month).
(Optional: $e^{-0.08} \approx 0.92$ → every additional month reduces the odds of churning by about 8 %.)

b) TP = 120, FP = 200 − 120 = 80, FN = 180 − 120 = 60, TN = 1000 − 120 − 80 − 60 = 740.
Accuracy = 860/1000 = 0.86; Precision = 120/200 = 0.60; Recall = 120/180 ≈ 0.667; FPR = 80/820 ≈ 0.098;
F1 = 2·0.6·0.667/(0.6+0.667) ≈ 0.632.

c) Always predicting "no churn" gives an accuracy of 820/1000 = **0.82** – almost as high as the model – but a recall of
**0**: not a single churner is found. This is why accuracy alone misleads with imbalanced classes.

## Exercise 2: Import and prepare the data

a) Import the data. The column `TotalCharges` is stored as text and contains a few blanks. Convert it to a number
(`pd.to_numeric(..., errors='coerce')`) and look at the rows with missing values. What do they have in common? Remove
them.

b) Create the binary target `churn` (1 = Yes, 0 = No). What is the churn rate?

c) Plot the churn rate by `Contract` and by `InternetService`. Interpret.

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/telco_customer_churn.csv', sep=',')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges:', df['TotalCharges'].isna().sum())
print(df.loc[df['TotalCharges'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])
df = df.dropna().reset_index(drop=True)
print(df.shape)

# b) Binary target and churn rate
df['churn'] = (df['Churn'] == 'Yes').astype(int)
print('Churn rate:', round(df['churn'].mean(), 3))

# c) Churn rate by contract and internet service
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, col in zip(axes, ['Contract', 'InternetService']):
    rate = df.groupby(col)['churn'].mean().sort_values()
    rate.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Churn rate')
    ax.set_title(f'Churn rate by {col}')
    for i, v in enumerate(rate):
        ax.text(v + 0.005, i, f'{v:.0%}', va='center')
plt.show()

**Solution:**

a) 11 customers have no `TotalCharges`; all of them have **tenure = 0** – they have just signed and have not been billed
yet. They are removed (7'032 customers remain).

b) The churn rate is about **26.6 %** → the classes are imbalanced.

c) Customers with a **month-to-month** contract churn far more often (≈ 43 %) than customers with a one-year (≈ 11 %) or
two-year contract (≈ 3 %). Customers with **fiber optic** internet churn more often (≈ 42 %) than DSL customers (≈ 19 %)
or customers without internet (≈ 7 %).

## Exercise 3: Logistic regression with one feature

a) Split the data into training (80 %) and test set (20 %) with `stratify=df['churn']` and `random_state=42`
(split the whole data frame: `train_df, test_df = train_test_split(df, ...)`).

b) Fit a logistic regression `churn ~ tenure` on the training set. Print $\hat\beta_0$ and $\hat\beta_1$ and compare
them with the coefficients of Exercise 1a.

c) Plot the predicted probability against `tenure` (0–72 months) together with the observed churn rate per tenure
(dots). Interpret the S-curve.

In [ ]:
# a) Stratified train/test split
train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['churn'], random_state=42)
print('Training set:', train_df.shape, ' Test set:', test_df.shape)
print('Churn rate train / test:', round(train_df['churn'].mean(), 3), '/', round(test_df['churn'].mean(), 3))

# b) Logistic regression with tenure only
log_reg_1 = LogisticRegression()
log_reg_1.fit(train_df[['tenure']], train_df['churn'])
print('beta_0 =', round(log_reg_1.intercept_[0], 3), ' beta_1 =', round(log_reg_1.coef_[0][0], 4))

# c) Plot predicted probability and observed churn rate per tenure
tenure_grid = pd.DataFrame({'tenure': np.arange(0, 73)})
observed = train_df.groupby('tenure')['churn'].mean()

plt.figure(figsize=(8, 4))
plt.scatter(observed.index, observed.values, s=15, color='grey', label='observed churn rate per tenure')
plt.plot(tenure_grid['tenure'], log_reg_1.predict_proba(tenure_grid)[:, 1], color='darkred',
         linewidth=2, label='logistic regression')
plt.axhline(0.5, color='black', linestyle=':', linewidth=1, label='threshold 0.5')
plt.xlabel('Tenure [months]')
plt.ylabel('P(churn)')
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

The estimated coefficients are close to the values of Exercise 1a ($\hat\beta_0 \approx 0$, $\hat\beta_1 \approx -0.04$
– the data-driven slope is flatter). The churn probability is highest for new customers (≈ 50 %) and decreases with every
month. The curve shows only the lower part of the "S", because tenure cannot be negative. The observed churn rates of new
customers (first months) are even higher than the curve – a single linear score cannot capture everything.

## Exercise 4: Logistic regression with all features

a) Use all features except `customerID`, `Churn` and `churn`. Create dummy variables for the categorical features with
`pd.get_dummies(..., drop_first=True, dtype=int)` and standardise the features with a `StandardScaler` (fit only on the
training set!).

b) Fit a `LogisticRegression(max_iter=1000)` and plot the 10 largest positive and 10 largest negative coefficients.
Since the features are standardised, the coefficients can be compared with each other. Which characteristics increase
and which decrease the churn probability?

In [ ]:
# a) Features, dummy variables and scaling
features = [c for c in df.columns if c not in ['customerID', 'Churn', 'churn']]
X_train = pd.get_dummies(train_df[features], drop_first=True, dtype=int)
X_test  = pd.get_dummies(test_df[features],  drop_first=True, dtype=int)
y_train, y_test = train_df['churn'], test_df['churn']
print('Number of features after dummy coding:', X_train.shape[1])

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# b) Fit the model and plot the coefficients
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

coefs = pd.Series(log_reg.coef_[0], index=X_train.columns).sort_values()
top = pd.concat([coefs.head(10), coefs.tail(10)])
plt.figure(figsize=(8, 6))
top.plot(kind='barh', color=np.where(top > 0, 'darkred', 'steelblue'))
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Coefficient (standardised features)')
plt.title('Logistic regression: largest coefficients')
plt.show()

**Solution:**

**Positive coefficients (higher churn probability):** fiber optic internet, high `TotalCharges` (given the tenure),
streaming TV/movies, multiple lines, payment by electronic check, paperless billing.

**Negative coefficients (lower churn probability):** long `tenure` (strongest effect), two- and one-year contracts,
online security, tech support, dependents.

Surprisingly, `MonthlyCharges` has a clearly **negative** coefficient, although customers with higher monthly charges
churn more often. The reason is **multicollinearity**: the monthly charges are almost completely determined by the booked
services (fiber optic, streaming, …), which are also in the model, and TotalCharges ≈ tenure × MonthlyCharges.
"Higher monthly charges *holding all services, tenure and total charges constant*" is hardly a meaningful scenario.
Moreover, several dummies are redundant ("No internet service" appears in seven columns). Single coefficients must
therefore be interpreted with care – the predictions are not affected.

## Exercise 5: Evaluation with the default threshold τ = 0.5

a) Predict the test set with `predict()` (= threshold 0.5). Plot the confusion matrix and compute accuracy, precision,
recall, F1 and FPR.

b) Compare the accuracy with the baseline "always no churn". Is the model useful? Which metric is most relevant for the
marketing team?

In [ ]:
# a) Confusion matrix and metrics
y_pred = log_reg.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred, labels=[1, 0])
TN, FP, FN, TP = confusion_matrix(y_test, y_pred).ravel()

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['churn', 'no churn'], yticklabels=['churn', 'no churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix (tau = 0.5)')
plt.show()

print(f'Accuracy  = {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision = {precision_score(y_test, y_pred):.3f}')
print(f'Recall    = {recall_score(y_test, y_pred):.3f}')
print(f'F1        = {f1_score(y_test, y_pred):.3f}')
print(f'FPR       = {FP / (FP + TN):.3f}')

# b) Baseline
print(f'\nBaseline accuracy (always no churn) = {1 - y_test.mean():.3f}')

**Solution:**

Note that the confusion matrix here (as on the slides) shows the actual classes in the rows and has the positive class
first.

The model reaches an accuracy of about **0.80** – only about 7 percentage points better than the baseline (≈ 0.73).
Precision is about 0.65 (two out of three flagged customers really churn), but the **recall is only about 0.58**: almost
half of the churners are missed. For the marketing team, recall (finding as many churners as possible) is usually most
important, as long as the number of unnecessary offers (FP) stays acceptable. The threshold 0.5 is not necessarily the
best choice → Exercise 6.

## Exercise 6: Choosing the threshold based on costs

The marketing team estimates:
- a **missed churner** (FN) costs **500 USD** (lost margin),
- an **unnecessary retention offer** (FP) costs **100 USD** (discount for a customer who would have stayed anyway).

a) Compute precision, recall, FPR and the total costs $500\cdot FN + 100\cdot FP$ on the test set for the thresholds
$\tau = 0.05, 0.10, \dots, 0.90$ (use `predict_proba`). Show the results as a table and plot the costs against $\tau$.

b) Which threshold minimises the costs? How do recall and precision change compared to $\tau = 0.5$?

*Note: strictly speaking, the threshold should be chosen on validation data (or with CV); here we use the test set to keep
the exercise short.*

In [ ]:
y_prob = log_reg.predict_proba(X_test_s)[:, 1]
cost_fn, cost_fp = 500, 100

rows = []
for tau in np.arange(0.05, 0.91, 0.05):
    y_pred_tau = (y_prob >= tau).astype(int)
    TN, FP, FN, TP = confusion_matrix(y_test, y_pred_tau).ravel()
    rows.append({'tau': round(tau, 2), 'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
                 'precision': precision_score(y_test, y_pred_tau, zero_division=0),
                 'recall': TP / (TP + FN), 'FPR': FP / (FP + TN),
                 'costs': cost_fn * FN + cost_fp * FP})
thresholds = pd.DataFrame(rows)
print(thresholds.round(3).to_string(index=False))

best = thresholds.loc[thresholds['costs'].idxmin()]
print(f"\nCost-optimal threshold: tau = {best['tau']}, costs = {best['costs']:,.0f} USD")
print(f"Costs at tau = 0.5:      {thresholds.loc[np.isclose(thresholds['tau'], 0.5), 'costs'].iloc[0]:,.0f} USD")

plt.figure(figsize=(7, 4))
plt.plot(thresholds['tau'], thresholds['costs'], marker='o')
plt.axvline(best['tau'], color='darkred', linestyle='--', label=f"minimum at tau = {best['tau']}")
plt.xlabel('Threshold tau')
plt.ylabel('Total costs on the test set [USD]')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

The cost-optimal threshold is much lower than 0.5: **τ = 0.2** (0.15 is almost equally good). Because a missed churner
costs five times as much as an unnecessary offer, it pays off to flag many more customers: the recall rises from ≈ 0.58
to ≈ 0.87, while precision drops from ≈ 0.65 to ≈ 0.46 and the FPR rises from ≈ 0.11 to ≈ 0.37. The total costs decrease
from about 91'000 to 63'000 USD (−31 %) compared to $\tau = 0.5$.

Rule of thumb: if FN are $c$ times as expensive as FP, the optimal threshold of a well-calibrated model is about
$\tau^* = 1/(1+c) = 1/6 \approx 0.17$.

## Exercise 7: ROC curve and AUC

a) Plot the ROC curve of the model on the test set (`roc_curve`) and compute the AUC (`roc_auc_score`). Mark the points
for $\tau = 0.5$ and for the cost-optimal threshold from Exercise 6. For comparison, add the ROC curve of the simple model
`log_reg_1` (only `tenure`).

b) Check the interpretation of the AUC empirically: draw 100'000 random pairs (one churner, one non-churner) from the test
set and compute the share of pairs in which the churner has the higher predicted probability (ties count ½).

c) Interpret the AUC values.

In [ ]:
# a) ROC curves
y_prob_1 = log_reg_1.predict_proba(test_df[['tenure']])[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
fpr_1, tpr_1, _ = roc_curve(y_test, y_prob_1)
auc_all, auc_1 = roc_auc_score(y_test, y_prob), roc_auc_score(y_test, y_prob_1)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'all features (AUC = {auc_all:.3f})')
plt.plot(fpr_1, tpr_1, linewidth=2, label=f'tenure only (AUC = {auc_1:.3f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--', label='random ranking')
for tau, marker in [(0.5, 'o'), (best['tau'], 's')]:
    row = thresholds.loc[np.isclose(thresholds['tau'], tau)].iloc[0]
    plt.scatter(row['FPR'], row['recall'], s=80, marker=marker, color='black', zorder=3,
                label=f'tau = {tau}: TPR = {row["recall"]:.2f}, FPR = {row["FPR"]:.2f}')
plt.xlabel('False positive rate (FPR)')
plt.ylabel('True positive rate (recall)')
plt.title('ROC curves (test set)')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.show()

# b) AUC as probability of a correct ranking
rng = np.random.default_rng(42)
p_pos = y_prob[y_test.values == 1]
p_neg = y_prob[y_test.values == 0]
a = rng.choice(p_pos, 100_000)
b_ = rng.choice(p_neg, 100_000)
share = np.mean(a > b_) + 0.5 * np.mean(a == b_)
print(f'Share of correctly ranked pairs: {share:.3f}  vs. AUC = {auc_all:.3f}')

**Solution:**

a) The model with all features has an AUC of about **0.84**, the model with `tenure` only about **0.75**. Both are far
better than random (0.5). Moving the threshold from 0.5 to the cost-optimal value moves us along the ROC curve to the
upper right: higher recall, but also higher FPR. The AUC itself does not depend on the threshold.

b) The share of correctly ranked pairs agrees with the AUC (up to sampling error).

c) In about 84 % of all (churner, non-churner) pairs the churner gets the higher churn probability. This makes the model
well suited to **rank** customers for a retention campaign, e.g. "contact the 20 % of customers with the highest risk".

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')